# Жизненный цикл и карточка моделей

Периметр формируется только из Hive:

1. версии с `model_ver_prom_expl_flag = true`;
2. все версии категорий A и B;
3. детерминированные 50% категории C;
4. детерминированные 50% категории D;
5. E, пустые и прочие категории исключаются.

Все связанные выгрузки строятся только для итоговых `model_ver_sid`.
Внешние списки ID и промежуточные Hive-таблицы не используются.


In [ ]:
import os
import sys
from functools import reduce
from pathlib import Path

os.environ["SPARK_MAJOR_VERSION"] = "3.5.1"
os.environ["SPARK_HOME"] = "/usr/sdp/current/spark3.5.1-client/"
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
sys.path.insert(0, "/usr/sdp/current/spark3.5.1-client/python/")
sys.path.insert(0, "/usr/sdp/current/spark3.5.1-client/python/lib/py4j-0.10.9.7-src.zip")

import pandas as pd
from pyspark import SparkConf, StorageLevel
from pyspark.sql import DataFrame, SparkSession, Window, functions as F

conf = (
    SparkConf()
    .setAppName("model_lifecycle_and_card")
    .setMaster("yarn")
    .set("spark.executor.cores", "2")
    .set("spark.executor.memory", "6g")
    .set("spark.executor.memoryOverhead", "1g")
    .set("spark.driver.memory", "6g")
    .set("spark.driver.maxResultSize", "4g")
    .set("spark.dynamicAllocation.enabled", "true")
    .set("spark.dynamicAllocation.initialExecutors", "4")
    .set("spark.dynamicAllocation.maxExecutors", "12")
    .set("spark.dynamicAllocation.executorIdleTimeout", "120s")
    .set("spark.dynamicAllocation.cachedExecutorIdleTimeout", "600s")
    .set("spark.shuffle.service.enabled", "true")
    .set("spark.sql.parquet.int96RebaseModeInWrite", "CORRECTED")
    .set("spark.sql.parquet.writeLegacyFormat", "true")
    .set("spark.sql.parquet.compression.codec", "snappy")
)
spark = SparkSession.builder.config(conf=conf).enableHiveSupport().getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version)


## Настройки и структура результата


In [ ]:
SOURCE_DB = "prx_pri_custom_ris_l_library_custom_risk_model_library"
OUTPUT_DIR = Path.cwd().resolve()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CATEGORY_SAMPLE_SEED = "model-risk-sample-v1"
HALF_ROUNDING = "ceil"
HISTORY_CUTOFF = "2025-10-02 00:00:00"
MAX_EXCEL_ROWS = 1_048_575

TABLES = {
    "model": "t_model",
    "model_ver": "t_model_ver",
    "anlt": "t_model_ver_anlt_dtl",
    "prom": "t_model_ver_prom",
    "prom_link": "t_model_ver_prom_x_prom_implm",
    "busn_link": "t_model_ver_x_busn_task",
    "busn": "t_busn_task",
    "valid": "t_valid",
    "valid_it": "t_valid_it",
    "manual_link": "t_model_ver_x_montrg_manual",
    "manual_result": "t_montrg_manual_rslt",
    "auto_link": "t_model_ver_x_montrg_auto",
    "auto": "t_montrg_auto",
    "change": "t_ent_param_chg",
}
TABLES = {key: SOURCE_DB + "." + value for key, value in TABLES.items()}

CARD_COLUMNS = [
    ("model_name", "MODEL_NAME|Наименование модели"),
    ("model_rsk_flag", "MODEL_RSK_FLAG|Флаг риск-модели"),
    ("model_rsk_type_name", "MODEL_RSK_TYPE_NAME|Тип риска модели"),
    ("model_rsk_sgmnt_name", "MODEL_RSK_SGMNT_NAME|Наименование риск-сегмента модели"),
    ("model_type_name", "MODEL_TYPE_NAME|Тип модели"),
    ("model_subtype_name", "MODEL_SUBTYPE_NAME|Подтип модели"),
    ("model_code", "MODEL_CODE|Код модели"),
    ("model_conf_ctgry_name", "MODEL_CONF_CTGRY_NAME|Категория конфиденциальности информации о модели"),
    ("model_sel_secret_flag", "MODEL_SEL_SECRET_FLAG|Флаг коммерческой тайны"),
    ("model_sid", "MODEL_SID|Идентификатор модели"),
    ("model_ver_signfcnt_lvl_name", "MODEL_VER_SIGNFCNT_LVL_NAME|Наименование степени значимости версии модели"),
    ("model_ver_dev_start_fact_dttm", "MODEL_VER_DEV_START_FACT_DTTM|Фактическая дата-время начала разработки версии модели"),
    ("model_ver_signfcnt_ctgry_code", "MODEL_VER_SIGNFCNT_CTGRY_CODE|Наименование категории значимости версии модели"),
    ("model_ver_signfcnt_ctgry_descr_txt", "MODEL_VER_SIGNFCNT_CTGRY_DESCR_TXT|Обоснование категории значимости версии модели"),
    ("model_ver_dev_end_fact_dttm", "MODEL_VER_DEV_END_FACT_DTTM|Фактическая дата-время окончания разработки версии модели"),
    ("model_ver_dev_sys_name", "MODEL_VER_DEV_SYS_NAME|Система в которой разрабатывалась версия модели"),
    ("model_ver_data_mart_link_txt", "MODEL_VER_DATA_MART_LINK_TXT|Ссылка на витрину данных для обучения версии модели"),
    ("model_ver_crtn_dttm", "MODEL_VER_CRTN_DTTM|Дата-время создания версии модели"),
    ("model_ver_dev_report_sid", "MODEL_VER_DEV_REPORT_SID|Идентификатор отчета о разработке версия модели"),
    ("model_ver_prevalid_report_file_sid", "MODEL_VER_PREVALID_REPORT_FILE_SID|Идентификатор файла с отчетом о превалидации версии модели"),
    ("model_ver_prevalid_report_link_sid", "MODEL_VER_PREVALID_REPORT_LINK_SID|Идентификатор ссылки на отчет о превалидации версии модели"),
    ("model_ver_prevalid_report_link_txt", "MODEL_VER_PREVALID_REPORT_LINK_TXT|Ссылка на отчет о превалидации"),
    ("model_ver_dev_block_name", "MODEL_VER_DEV_BLOCK_NAME|Наименование блока разработки версии модели"),
    ("model_ver_dev_dprtmt_name", "MODEL_VER_DEV_DPRTMT_NAME|Наименование подразделения разработки версии модели"),
    ("model_ver_sid", "MODEL_VER_SID|Идентификатор версии модели"),
    ("model_ver_stts_name", "MODEL_VER_STTS_NAME|Наименование статуса версии модели"),
    ("model_stts_name", "MODEL_STTS_NAME|Статус модели"),
    ("model_ver_prom_expl_flag", "MODEL_VER_PROM_EXPL_FLAG|Флаг нахождения версии модели в эксплуатации"),
    ("busn_task_sid", "BUSN_TASK_SID|Идентификатор бизнес-задачи"),
    ("model_ver_prom_instr_name", "MODEL_VER_PROM_INSTR_NAME|Среда (инструмент) исполнения промышленной версии модели"),
    ("model_ver_prom_sys_name", "MODEL_VER_PROM_SYS_NAME|Система в которой реализована промышленная версия модели"),
    ("model_ver_prom_sid", "MODEL_VER_PROM_SID|Идентификатор промышленной версии модели"),
    ("model_ver_prom_stts_name", "MODEL_VER_PROM_STTS_NAME|Статус промышленной версии модели"),
    ("model_ver_prom_crtn_dttm", "MODEL_VER_PROM_CRTN_DTTM|Дата-время создания промышленной версии модели"),
    ("model_ver_prom_dev_block_name", "MODEL_VER_PROM_DEV_BLOCK_NAME|Блок разработки промышленной версии модели"),
    ("model_ver_prom_dev_dprtmt_name", "MODEL_VER_PROM_DEV_DPRTMT_NAME|Подразделение разработки промышленной версии модели"),
    ("montrg_manual_sid", "MONTRG_MANUAL_SID|Идентификатор ручного мониторинга"),
    ("montrg_auto_sid", "MONTRG_AUTO_SID|Идентификатор автоматического мониторинга"),
    ("valid_crtn_dttm", "VALID_CRTN_DTTM|Дата-время создания валидации"),
    ("valid_sid", "VALID_SID|Идентификатор валидации"),
    ("valid_report_sid", "VALID_REPORT_SID|Идентификатор отчета о валидации"),
    ("valid_dprtmt_name", "VALID_DPRTMT_NAME|Подразделение, проводящее валидацию"),
    ("busn_task_name", "BUSN_TASK_NAME|Наименование бизнес-задачи"),
    ("busn_task_claim_descr_txt", "BUSN_TASK_CLAIM_DESCR_TXT|Описание требований бизнес-задачи"),
    ("busn_task_crtn_dttm", "BUSN_TASK_CRTN_DTTM|Дата-время создания бизнес-задачи"),
    ("busn_task_employer_block_name", "BUSN_TASK_EMPLOYER_BLOCK_NAME|Блок заказчика бизнес-задачи"),
    ("busn_task_employer_dprtmt_name", "BUSN_TASK_EMPLOYER_DPRTMT_NAME|Наименование подразделения заказчика бизнес-задачи"),
    ("montrg_manual_rslt_end_dttm", "MONTRG_MANUAL_RSLT_END_DTTM|Дата-время окончания процесса ручного мониторинга в рамках которого получен текущий результат"),
    ("montrg_manual_rslt_start_dttm", "MONTRG_MANUAL_RSLT_START_DTTM|Дата-время начала процесса ручного мониторинга в рамках которого получен текущий результат"),
    ("montrg_manual_rslt_report_sid", "MONTRG_MANUAL_RSLT_REPORT_SID|Идентификатор отчёта о результате ручного мониторинга"),
    ("montrg_auto_crtn_dttm", "MONTRG_AUTO_CRTN_DTTM|Дата-время создания автоматического мониторинга"),
    ("montrg_auto_end_dttm", "MONTRG_AUTO_END_DTTM|Дата-время окончания мониторинга"),
    ("montrg_auto_dprtmt_name", "MONTRG_AUTO_DPRTMT_NAME|Наименование подразделения проводящего автоматического мониторинга"),
    ("valid_it_end_dttm", "VALID_IT_END_DTTM|Дата-время окончания ИТ-валидации"),
    ("valid_it_start_dttm", "VALID_IT_START_DTTM|Дата-время начала ИТ-валидации"),
    ("valid_it_crtn_dttm", "VALID_IT_CRTN_DTTM|Дата-время создания ИТ-валидации"),
    ("valid_it_rslt_name", "VALID_IT_RSLT_NAME|Результат ИТ-валидации"),
    ("valid_it_dprtmt_name", "VALID_IT_DPRTMT_NAME|Подразделение проводящее ИТ-валидацию"),
]

CARD_MODEL_VER_COLUMN = "MODEL_VER_SID|Идентификатор версии модели"
CARD_MODEL_COLUMN = "MODEL_SID|Идентификатор модели"
CATEGORY_COLUMN = "MODEL_VER_SIGNFCNT_CTGRY_CODE|Наименование категории значимости версии модели"
EXPLOITATION_FLAG_COLUMN = "MODEL_VER_PROM_EXPL_FLAG|Флаг нахождения версии модели в эксплуатации"


## Чтение источников и построение карточки


In [ ]:
def require_columns(df: DataFrame, table_name: str, columns) -> None:
    missing = sorted(set(columns) - set(df.columns))
    if missing:
        raise RuntimeError("{}: отсутствуют поля: {}".format(table_name, ", ".join(missing)))


def source(name: str, columns) -> DataFrame:
    table_name = TABLES[name]
    if not spark.catalog.tableExists(table_name):
        raise RuntimeError("Не найдена таблица: " + table_name)
    df = spark.table(table_name)
    require_columns(df, table_name, columns)
    return df.select(*columns)


def latest(df: DataFrame, keys, order_column: str = "start_dt") -> DataFrame:
    window = Window.partitionBy(*keys).orderBy(F.col(order_column).desc())
    return df.withColumn("__rn", F.row_number().over(window)).filter(F.col("__rn") == 1).drop("__rn")


def latest_source(name: str, keys, columns) -> DataFrame:
    selected = list(dict.fromkeys(list(keys) + list(columns) + ["start_dt"]))
    return latest(source(name, selected), keys).drop("start_dt")


model = latest_source("model", ["model_sid"], [
    "model_name", "model_rsk_flag", "model_rsk_type_name", "model_rsk_sgmnt_name",
    "model_type_name", "model_subtype_name", "model_code", "model_conf_ctgry_name",
    "model_sel_secret_flag",
])
model_ver = latest_source("model_ver", ["model_sid"], [
    "model_ver_sid", "model_ver_signfcnt_lvl_name", "model_ver_dev_start_fact_dttm",
    "model_ver_signfcnt_ctgry_code", "model_ver_signfcnt_ctgry_descr_txt",
    "model_ver_dev_end_fact_dttm", "model_ver_dev_sys_name", "model_ver_data_mart_link_txt",
    "model_ver_crtn_dttm", "model_ver_dev_report_sid", "model_ver_prevalid_report_file_sid",
    "model_ver_prevalid_report_link_sid", "model_ver_prevalid_report_link_txt",
    "model_ver_dev_block_name", "model_ver_dev_dprtmt_name",
])
anlt = latest_source("anlt", ["model_ver_sid"], [
    "model_ver_stts_name", "model_stts_name", "model_ver_prom_expl_flag",
])
prom = latest_source("prom", ["model_ver_sid"], [
    "model_ver_prom_sid", "model_ver_prom_instr_name", "model_ver_prom_sys_name",
    "model_ver_prom_stts_name", "model_ver_prom_crtn_dttm",
    "model_ver_prom_dev_block_name", "model_ver_prom_dev_dprtmt_name",
])
busn_link = latest_source("busn_link", ["model_ver_sid"], ["busn_task_sid"])
manual_link = latest_source("manual_link", ["model_ver_sid"], ["montrg_manual_sid"])
auto_link = latest_source("auto_link", ["model_ver_sid"], ["montrg_auto_sid"])
valid = latest_source("valid", ["model_ver_sid"], [
    "valid_sid", "valid_crtn_dttm", "valid_report_sid", "valid_dprtmt_name",
])
busn = latest_source("busn", ["busn_task_sid"], [
    "busn_task_name", "busn_task_claim_descr_txt", "busn_task_crtn_dttm",
    "busn_task_employer_block_name", "busn_task_employer_dprtmt_name",
])
manual_result = latest_source("manual_result", ["montrg_manual_sid"], [
    "montrg_manual_rslt_end_dttm", "montrg_manual_rslt_start_dttm",
    "montrg_manual_rslt_report_sid",
])
auto = latest_source("auto", ["montrg_auto_sid"], [
    "montrg_auto_crtn_dttm", "montrg_auto_end_dttm", "montrg_auto_dprtmt_name",
])
valid_it = latest_source("valid_it", ["model_ver_prom_sid"], [
    "valid_it_sid", "valid_it_end_dttm", "valid_it_start_dttm", "valid_it_crtn_dttm",
    "valid_it_rslt_name", "valid_it_dprtmt_name",
])

card = (
    model_ver.join(model, "model_sid", "inner")
    .join(anlt, "model_ver_sid", "left")
    .join(busn_link, "model_ver_sid", "left")
    .join(prom, "model_ver_sid", "left")
    .join(manual_link, "model_ver_sid", "left")
    .join(auto_link, "model_ver_sid", "left")
    .join(valid, "model_ver_sid", "left")
    .join(busn, "busn_task_sid", "left")
    .join(manual_result, "montrg_manual_sid", "left")
    .join(auto, "montrg_auto_sid", "left")
    .join(valid_it, "model_ver_prom_sid", "left")
)
card_full = card.select(*[F.col(source_name).alias(output_name) for source_name, output_name in CARD_COLUMNS])
print("Карточка сформирована")


## Эксплуатация → A/B полностью → 50% C → 50% D


In [ ]:
def half_size(size: int) -> int:
    if HALF_ROUNDING == "ceil":
        return (size + 1) // 2
    if HALF_ROUNDING == "floor":
        return size // 2
    raise ValueError("HALF_ROUNDING должен быть 'ceil' или 'floor'")


def to_pandas_limited(df: DataFrame, name: str) -> pd.DataFrame:
    result = df.limit(MAX_EXCEL_ROWS + 1).toPandas()
    if len(result) > MAX_EXCEL_ROWS:
        raise RuntimeError(name + ": превышен лимит строк Excel")
    return result


card_full_count = card_full.count()
duplicate_count = card_full.groupBy(F.col("`" + CARD_MODEL_VER_COLUMN + "`")).count().filter("count > 1").count()
if duplicate_count:
    raise RuntimeError("Карточка содержит дубликаты model_ver_sid: {}".format(duplicate_count))

operational = (
    card_full
    .filter(F.col("`" + EXPLOITATION_FLAG_COLUMN + "`").cast("boolean") == F.lit(True))
    .withColumn("__category", F.upper(F.trim(F.col("`" + CATEGORY_COLUMN + "`").cast("string"))))
    .persist(StorageLevel.MEMORY_AND_DISK)
)
operational_count = operational.count()
if not operational_count:
    raise RuntimeError("Не найдено версий в эксплуатации")

category_before = {row["__category"]: int(row["count"]) for row in operational.groupBy("__category").count().collect()}
candidate_counts = {key: category_before.get(key, 0) for key in ("A", "B", "C", "D")}
keep_counts = {
    "A": candidate_counts["A"],
    "B": candidate_counts["B"],
    "C": half_size(candidate_counts["C"]),
    "D": half_size(candidate_counts["D"]),
}

sample_window = Window.partitionBy("__category").orderBy(
    F.xxhash64(F.col("`" + CARD_MODEL_COLUMN + "`").cast("string"), F.lit(CATEGORY_SAMPLE_SEED)),
    F.col("`" + CARD_MODEL_COLUMN + "`").cast("string"),
    F.col("`" + CARD_MODEL_VER_COLUMN + "`").cast("string"),
)
selected_models_sdf = (
    operational
    .filter(F.col("__category").isin("A", "B", "C", "D"))
    .withColumn("__sample_rank", F.row_number().over(sample_window))
    .filter(
        F.col("__category").isin("A", "B")
        | ((F.col("__category") == "C") & (F.col("__sample_rank") <= F.lit(keep_counts["C"])))
        | ((F.col("__category") == "D") & (F.col("__sample_rank") <= F.lit(keep_counts["D"])))
    )
    .drop("__sample_rank")
    .persist(StorageLevel.MEMORY_AND_DISK)
)
selected_count = selected_models_sdf.count()
if not selected_count:
    raise RuntimeError("Итоговый периметр пуст")

selected_by_category = {
    row["__category"]: int(row["count"])
    for row in selected_models_sdf.groupBy("__category").count().collect()
}
for category in ("A", "B", "C", "D"):
    if selected_by_category.get(category, 0) != keep_counts[category]:
        raise RuntimeError("Нарушен отбор категории " + category)

model_sample_sdf = (
    selected_models_sdf
    .select(F.col("`" + CARD_MODEL_VER_COLUMN + "`").cast("string").alias("model_ver_sid"))
    .dropDuplicates(["model_ver_sid"])
    .persist(StorageLevel.MEMORY_AND_DISK)
)
model_sample_count = model_sample_sdf.count()
if model_sample_count != selected_count:
    raise RuntimeError("Количество уникальных model_ver_sid не совпало с итоговым периметром")

output_pdf = to_pandas_limited(selected_models_sdf.drop("__category"), "output.xlsx")
output_pdf.to_excel(OUTPUT_DIR / "output.xlsx", index=False)
pd.DataFrame(
    [(column, int(output_pdf[column].isna().sum())) for column in output_pdf.columns],
    columns=["Key", "Value"],
).to_excel(OUTPUT_DIR / "example.xlsx", index=False)

category_selection_control_pdf = pd.DataFrame({
    "category": ["A", "B", "C", "D", "E", "EMPTY_OR_OTHER"],
    "operational_count": [
        category_before.get("A", 0), category_before.get("B", 0),
        category_before.get("C", 0), category_before.get("D", 0),
        category_before.get("E", 0),
        operational_count - sum(category_before.get(x, 0) for x in ("A", "B", "C", "D", "E")),
    ],
    "selected_count": [
        selected_by_category.get("A", 0), selected_by_category.get("B", 0),
        selected_by_category.get("C", 0), selected_by_category.get("D", 0), 0, 0,
    ],
    "expected_selected": [
        keep_counts["A"], keep_counts["B"], keep_counts["C"], keep_counts["D"], 0, 0,
    ],
})
selected_model_ids_pdf = to_pandas_limited(model_sample_sdf.orderBy("model_ver_sid"), "selected_model_ids")

print("Карточка до отбора:", card_full_count)
print("Версий в эксплуатации:", operational_count)
print(category_selection_control_pdf.to_string(index=False))
print("Итоговый периметр:", selected_count)


## Истории статусов и параметры итогового периметра


In [ ]:
def map_values(column_name: str, mapping):
    items = []
    for source_value, target_value in mapping.items():
        items.extend([F.lit(source_value), F.lit(target_value)])
    return F.coalesce(F.element_at(F.create_map(*items), F.col(column_name)), F.col(column_name))


def cap_open_end(column_name: str):
    return F.when(
        F.col(column_name).cast("string") == "9999-12-31 00:00:00",
        F.current_timestamp(),
    ).otherwise(F.col(column_name).cast("timestamp"))


def selected_latest(source_df: DataFrame, output_columns) -> DataFrame:
    joined = (
        model_sample_sdf.alias("s")
        .join(source_df.alias("x"), F.col("s.model_ver_sid") == F.col("x.model_ver_sid").cast("string"), "left")
        .select(F.col("s.model_ver_sid"), *[F.col("x." + c) for c in output_columns], F.col("x.start_dt").alias("__start_dt"))
    )
    return latest(joined, ["model_ver_sid"], "__start_dt").drop("__start_dt")


change_events = source("change", [
    "ent_sid", "ent_type_name", "ent_param_chg_sid", "ent_param_chg_val",
    "ent_param_chg_usr_name", "start_dttm", "end_dttm",
]).persist(StorageLevel.MEMORY_AND_DISK)
busn_link_history = source("busn_link", ["model_ver_sid", "busn_task_sid", "start_dt"])
prom_history = source("prom", ["model_ver_sid", "model_ver_prom_sid", "model_ver_prom_stts_name", "start_dt"])
prom_link_history = source("prom_link", ["model_ver_prom_sid", "prom_implm_sid", "start_dt"])
model_ver_history = source("model_ver", ["model_ver_sid", "model_ver_prevalid_dttm"])
valid_history = source("valid", ["model_ver_sid", "valid_sid", "start_dt"])
valid_it_history = source("valid_it", ["model_ver_prom_sid", "valid_it_sid", "start_dt"])
auto_link_history = source("auto_link", ["model_ver_sid", "montrg_auto_sid", "start_dt"])

selected_busn = selected_latest(busn_link_history, ["busn_task_sid"])
selected_prom = selected_latest(
    prom_history.filter(F.col("model_ver_prom_stts_name") == "MODEL_PROM_VERSION_EXPLOIT_PERMITTED"),
    ["model_ver_prom_sid"],
)
selected_implm = latest(
    selected_prom.alias("p")
    .join(prom_link_history.alias("x"), F.col("p.model_ver_prom_sid") == F.col("x.model_ver_prom_sid"), "left")
    .select("p.model_ver_sid", F.col("x.prom_implm_sid"), F.col("x.start_dt").alias("__start_dt")),
    ["model_ver_sid"],
    "__start_dt",
).drop("__start_dt")

business_status_labels = {
    "BUSINESS_TASK_BACKLOG": "Ожидает начала (бэклог)",
    "BUSINESS_TASK_DEVELOPMENT": "В работе",
    "BUSINESS_TASK_DECISION_MAKING": "Принятие решения о завершении задачи",
    "BUSINESS_TASK_DONE": "Завершена",
}
implementation_status_labels = {
    "IMPLEMENTATION_PREPARATION_PILOT": "Подготовка к пилоту",
    "IMPLEMENTATION_PREPARATION_EXPLOITATION": "Подготовка в эксплуатации",
    "IMPLEMENTATION_EXPLOITATION": "Эксплуатация",
    "IMPLEMENTATION_PILOT_DONE": "Пилот завершен",
    "IMPLEMENTATION_WAITING_VALIDATION": "Ожидает валидации",
    "IMPLEMENTATION_EXPLOITATION_WITHOUT_VALIDATION": "Эксплуатация без валидации",
    "IMPLEMENTATION_WAITING_IT_VALIDATION": "Ожидает IT-валидации",
    "IMPLEMENTATION_FORMATION": "Формирование",
    "IMPLEMENTATION_EXPLOITATION_WITHDRAWN": "Выведено из эксплуатации",
    "IMPLEMENTATION_PILOT": "Пилот",
    "IMPLEMENTATION_CANCELED": "Отменено",
}
cutoff = F.to_timestamp(F.lit(HISTORY_CUTOFF))

business_task_input_sdf = selected_busn.filter("busn_task_sid is not null").select(F.col("busn_task_sid").cast("string")).distinct()
implementation_input_sdf = selected_implm.filter("prom_implm_sid is not null").select(F.col("prom_implm_sid").cast("string").alias("implm_sid")).distinct()

busn_task_life_stage = (
    business_task_input_sdf.alias("ids")
    .join(
        change_events.filter(
            (F.col("ent_type_name") == "BUSINESS_TASK")
            & (F.col("start_dttm") < cutoff)
            & (F.col("ent_param_chg_sid") == "BUSINESS_TASK_STATUS")
        ).alias("c"),
        F.col("ids.busn_task_sid") == F.col("c.ent_sid").cast("string"),
        "left",
    )
    .select(
        "ids.busn_task_sid",
        map_values("c.ent_param_chg_val", business_status_labels).alias("life_stage_name"),
        F.col("c.start_dttm").alias("start_dttm"),
        cap_open_end("c.end_dttm").alias("end_dttm"),
    )
)
implm_life_stage = (
    implementation_input_sdf.alias("ids")
    .join(
        change_events.filter(
            (F.col("start_dttm") < cutoff)
            & (F.col("ent_param_chg_sid") == "IMPLEMENTATION_STATUS")
        ).alias("c"),
        F.col("ids.implm_sid") == F.col("c.ent_sid").cast("string"),
        "left",
    )
    .select(
        "ids.implm_sid",
        map_values("c.ent_param_chg_val", implementation_status_labels).alias("life_stage_name"),
        F.col("c.start_dttm").alias("start_dttm"),
        cap_open_end("c.end_dttm").alias("end_dttm"),
    )
)

def latest_parameter(parameter_sid: str, value_alias: str, with_author: bool = False) -> DataFrame:
    fields = [F.col("s.model_ver_sid"), F.col("c.ent_param_chg_val").alias(value_alias)]
    if with_author:
        fields.extend([
            F.col("c.ent_param_chg_usr_name").alias("chg_usr_name"),
            F.col("c.start_dttm").alias("data"),
        ])
    joined = (
        model_sample_sdf.alias("s")
        .join(
            change_events.filter(F.col("ent_param_chg_sid") == parameter_sid).alias("c"),
            F.col("s.model_ver_sid") == F.col("c.ent_sid").cast("string"),
            "inner",
        )
        .select(*fields, F.col("c.start_dttm").alias("__start_dt"))
    )
    return latest(joined, ["model_ver_sid"], "__start_dt").drop("__start_dt")


old_importance = latest_parameter("MODEL_VERSION_IMPORTANCE_UMR", "old_import_umr")
new_importance = latest_parameter("MODEL_VERSION_MODEL_EFFECT", "new_import_umr", True)
model_with_old_new_importance = (
    new_importance.join(old_importance, "model_ver_sid", "left")
    .select("model_ver_sid", "old_import_umr", "new_import_umr", "chg_usr_name", "data")
)

print("Связанных business_task_sid:", business_task_input_sdf.count())
print("Связанных implm_sid:", implementation_input_sdf.count())


## Временная шкала итогового периметра


In [ ]:
def change_stage(base: DataFrame, entity_column: str, parameter_sids, stage_name: str) -> DataFrame:
    params = [parameter_sids] if isinstance(parameter_sids, str) else list(parameter_sids)
    return (
        base.alias("b")
        .join(
            change_events.filter(F.col("ent_param_chg_sid").isin(*params)).alias("c"),
            F.col("b." + entity_column).cast("string") == F.col("c.ent_sid").cast("string"),
            "inner",
        )
        .groupBy(F.col("b.model_ver_sid"))
        .agg(F.min("c.start_dttm").alias("start_dt"), F.max("c.start_dttm").alias("end_dt"))
        .withColumn("life_cycle_stage", F.lit(stage_name))
        .select("model_ver_sid", "start_dt", "end_dt", "life_cycle_stage")
    )


stage_business = change_stage(selected_busn, "busn_task_sid", "BUSINESS_TASK_STATUS", "Постановка бизнес-задачи")
stage_development = change_stage(
    model_sample_sdf.select("model_ver_sid", F.col("model_ver_sid").alias("entity_sid")),
    "entity_sid", "MODEL_VERSION_STATUS", "Разработка модели",
)
stage_prevalid = (
    model_sample_sdf.alias("s")
    .join(model_ver_history.alias("v"), F.col("s.model_ver_sid") == F.col("v.model_ver_sid").cast("string"), "inner")
    .groupBy("s.model_ver_sid")
    .agg(F.min("v.model_ver_prevalid_dttm").alias("start_dt"), F.max("v.model_ver_prevalid_dttm").alias("end_dt"))
    .filter(F.col("start_dt").isNotNull() & F.col("end_dt").isNotNull())
    .withColumn("life_cycle_stage", F.lit("Превалидация модели"))
    .select(F.col("model_ver_sid"), "start_dt", "end_dt", "life_cycle_stage")
)
selected_valid = selected_latest(valid_history, ["valid_sid"])
stage_validation = change_stage(selected_valid, "valid_sid", "VALIDATION_STATUS", "Валидация модели")
selected_auto = selected_latest(auto_link_history, ["montrg_auto_sid"])
stage_auto = change_stage(selected_auto, "montrg_auto_sid", "MONITORING_STATUS", "Модель поставлена на автомониторинг")
stage_prom = change_stage(selected_prom, "model_ver_prom_sid", "MODEL_PROM_VERSION_STATUS", "Разработка промышленной версии модели")

selected_valid_it = latest(
    selected_prom.alias("p")
    .join(valid_it_history.alias("v"), F.col("p.model_ver_prom_sid") == F.col("v.model_ver_prom_sid"), "left")
    .select("p.model_ver_sid", F.col("v.valid_it_sid"), F.col("v.start_dt").alias("__start_dt")),
    ["model_ver_sid"], "__start_dt",
).drop("__start_dt")
stage_it_validation = change_stage(selected_valid_it, "valid_it_sid", "IT_VALIDATION_STATUS", "Начало IT-валидации")
stage_implementation = change_stage(
    selected_implm, "prom_implm_sid",
    ["IMPLEMENTATION_STATUS", "IMPLEMENTATION_PILOT_CRITERIA"],
    "Внедрение модели",
)

model_lifecycle = reduce(
    lambda left, right: left.unionByName(right),
    [
        stage_business, stage_development, stage_prevalid, stage_validation,
        stage_auto, stage_prom, stage_it_validation, stage_implementation,
    ],
)
lifecycle_bounds = model_lifecycle.groupBy("model_ver_sid").agg(
    F.min("start_dt").alias("min(start_dt)"),
    F.max("end_dt").alias("max(end_dt)"),
)


## Выгрузка и итоговый контроль


In [ ]:
def save_excel(df: DataFrame, filename: str) -> None:
    to_pandas_limited(df, filename).to_excel(OUTPUT_DIR / filename, index=False)
    print("Сохранён:", OUTPUT_DIR / filename)


save_excel(busn_task_life_stage, "busn_task_life_stage.xlsx")
save_excel(implm_life_stage, "df_implm_life_stage.xlsx")
save_excel(model_with_old_new_importance, "model_with_old_new_importance_umr.xlsx")
save_excel(model_lifecycle, "data_to_process_mining.xlsx")
save_excel(lifecycle_bounds, "data_life_stage.xlsx")

summary_pdf = pd.DataFrame([
    ("карточка до отбора", card_full_count),
    ("версии в эксплуатации", operational_count),
    ("итог после A/B + 50% C/D", selected_count),
    ("уникальные model_ver_sid", model_sample_count),
    ("E в результате", int((output_pdf[CATEGORY_COLUMN].astype("string").str.strip().str.upper() == "E").sum())),
], columns=["check", "value"])

with pd.ExcelWriter(OUTPUT_DIR / "selection_control.xlsx", engine="openpyxl") as writer:
    summary_pdf.to_excel(writer, sheet_name="summary", index=False)
    category_selection_control_pdf.to_excel(writer, sheet_name="category_selection", index=False)
    selected_model_ids_pdf.to_excel(writer, sheet_name="selected_model_ids", index=False)

for df in (change_events, model_sample_sdf, selected_models_sdf, operational):
    df.unpersist()

print("Сохранены output.xlsx, example.xlsx, пять связанных выгрузок и selection_control.xlsx")
print("Контроль отбора пройден: эксплуатация → все A/B → 50% C → 50% D; E/пустые/прочие исключены")


Spark-сессия намеренно не останавливается. Если после ноутбука она не нужна, выполните `spark.stop()` отдельно.
